# 리그 제한 해제 — 학습 데이터 확대 실험

가설: 다른 리그의 선수/팀 경기까지 학습하면 선수→팀 표현이 더 안정적으로 학습될 수 있다.

**비교 규칙:** 2025년 학습, 이전과 같은 2026년 LCK 186시리즈 평가. A/B/C 구조·13피처·seed42/43/44·Adam.001·max500/patience30은 유지한다. 2025 LCK 마지막 33시리즈를 동일하게 early stopping에 사용한다. 평가 결과로 epoch를 고르지 않는다.

## 이번에 바뀐 것과 고정한 것

- 학습 표본: LCK 555세트 → 식별·필수통계 검사를 통과한 모든 리그 8,859세트.
- 별도 리그 이름 필터는 없고, 게임마다 같은 가중치를 준다. 리그별 균형화나 상대 리그 강도 피처는 추가하지 않았다.
- **LCK 학습/검증/평가의 원래 입력은 유지**한다. 국제전까지 LCK rolling 입력에 섞으면 표본 확대와 입력 변경이 동시에 일어나므로 이번에는 분리했다.
- 추가 non-LCK 세트는 해당 선수 ID의 이전 적격 경기들을 리그 구분 없이 최근 5세트로 집계한다. 신인 보충/표준화는 fitting 데이터에서 추정한다.
- non-LCK는 세트 승패를 학습하므로 Bo1/Bo2도 사용할 수 있다. 그 리그들의 best_of를 억지로 복원하지 않는다.
- 기존과 같은 full-batch optimizer 업데이트 방식이다. 한 epoch의 경기 수가 늘어나며 학습 분포와 scaler도 확대 자료를 반영한다.
- 마지막 refit은 2025년 12월까지의 적격 경기를 사용한다. 2026년은 가중치·scaler 학습에 사용하지 않는다.

## 데이터 검사

원본 45개 리그/대회, 10,041세트.

- datacompleteness가 complete가 아닌 820세트 제외. LPL 805세트의 15분 골드/XP/CS 차이가 모두 결측이라 현 입력을 그대로 사용할 수 없다.
- complete지만 선수 ID가 없는 362세트 제외. 이름을 임의로 ID 대신 쓰지 않는다.
- 최종 44개 리그/대회 8,859세트. LCK 555 + 다른 리그 8,304.
- LCK 비중 6.26%, 총 표본 약 15.96배.

세트당 10명, 팀당 5개 포지션, 중복 player ID, 0/1 타깃·승자, 필요한 통계 유한성, 선수의 같은 시각 중복 경기를 검사한다. 원본을 수정하지 않는다.

## 해석 제한

2026년은 연구 과정에서 반복 열람한 회고 평가 자료다. 이 실험 자체는 2025년에서 모델을 고정하지만 untouched test는 아니다. 첫 세트 명단이 알려져 있고, 동일/독립 세트 확률을 Bo3/Bo5로 변환한다는 가정도 유지한다.


In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display
ROOT=Path('/Users/seungyunmok/Developer/LOL_ML')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
RESULTS=ROOT/'notebook/experiments/11_multileague'

## 재실행

True이면 데이터 검사부터 학습·평가까지 다시 실행한다. 기본값 False는 저장된 결과만 읽는다.

In [2]:
RUN_EXPERIMENT=False
if RUN_EXPERIMENT:
    from backend.training.player_multileague import run
    run()

## 리그별 사용/제외 내역

In [3]:
display(pd.read_csv(RESULTS/'league_audit_2025.csv'))
display(pd.read_csv(RESULTS/'training_counts.csv'))
print(json.dumps(json.loads((RESULTS/'lock.json').read_text()),indent=2))

,league,eligible,missing_player_id,not_complete
0,AL,278,15,0
1,ASI,42,0,0
2,Asia Master,177,0,0
3,CD,296,5,0
4,CT,50,0,0
5,DCup,75,0,3
6,EBL,191,0,0
7,EM,439,0,0
8,EWC,33,0,0
9,FST,35,0,0


,league,train_sets
0,AL,278
1,ASI,42
2,Asia Master,177
3,CD,296
4,CT,50
5,DCup,75
6,EBL,191
7,EM,439
8,EWC,33
9,FST,35


{
  "epochs": {
    "position_additive": {
      "42": 3,
      "43": 22,
      "44": 74
    },
    "cross_position": {
      "42": 61,
      "43": 13,
      "44": 1
    },
    "upper_lower": {
      "42": 62,
      "43": 71,
      "44": 25
    }
  },
  "train_sets": 8859,
  "training_leagues": 44,
  "extra_sets": 8304,
  "lck_sets": 555,
  "fit_sets": 6988,
  "validation_series": 33,
  "cutoff": "2025-08-21 00:00:00",
  "train_end": "2025-12-29 10:52:27",
  "parameter_counts": {
    "position_additive": 213,
    "cross_position": 237,
    "upper_lower": 225
  }
}


## 같은 2026년 LCK 평가 — 3개 seed 확률 앙상블

In [4]:
comparison=pd.read_csv(RESULTS/'comparison_2026.csv')
display(comparison[comparison.level.eq('series')].round(4))
display(comparison[comparison.level.eq('set')].round(4))

,regime,track,level,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
1,lck_only_2025,position_additive,series,186,118,0.6344,0.6484,0.6754,0.2412,0.4166,0.6090
3,lck_only_2025,cross_position,series,186,106,0.5699,0.6438,0.6755,0.2412,0.4193,0.5921
5,lck_only_2025,upper_lower,series,186,115,0.6183,0.6582,0.6757,0.2413,0.4188,0.5825
7,multi_league_2025,position_additive,series,186,112,0.6022,0.6399,0.6733,0.2401,0.3909,0.6302
9,multi_league_2025,cross_position,series,186,109,0.5860,0.6585,0.6757,0.2413,0.4213,0.5856
11,multi_league_2025,upper_lower,series,186,111,0.5968,0.6562,0.6816,0.2442,0.4501,0.5534


,regime,track,level,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,lck_only_2025,position_additive,set,497,294,0.5915,0.6206,0.6829,0.2449,0.4366,0.5703
2,lck_only_2025,cross_position,set,497,277,0.5573,0.6203,0.6831,0.2450,0.4409,0.5616
4,lck_only_2025,upper_lower,set,497,283,0.5694,0.6257,0.6837,0.2453,0.4446,0.5551
6,multi_league_2025,position_additive,set,497,293,0.5895,0.6155,0.6814,0.2441,0.4183,0.5847
8,multi_league_2025,cross_position,set,497,285,0.5734,0.6252,0.6835,0.2452,0.4415,0.5620
10,multi_league_2025,upper_lower,set,497,287,0.5775,0.6277,0.6868,0.2469,0.4661,0.5339


## 결과

| 모델 | LCK만 정확도 | 여러 리그 정확도 | 정답 수 변화 | LCK만 log loss | 여러 리그 log loss |
|---|---:|---:|---:|---:|---:|
| A 합산 | 63.44% | 60.22% | 118→112 | 0.6754 | 0.6733 |
| B 전체 상호작용 | 56.99% | 58.60% | 106→109 | 0.6755 | 0.6757 |
| C 상체/하체 | 61.83% | 59.68% | 115→111 | 0.6757 | 0.6816 |

A는 확률 손실이 조금 좋아졌으나 승자 정확도는 낮아졌다. B만 정확도가 소폭 올랐다. 주 지표인 정확도 기준 기존 LCK-only A 유지.

단순히 여러 리그를 같은 가중치로 섞는 방식이 일관된 개선을 주지는 않았다. LCK 비중 감소, 리그 간 분포 차이, 작은 validation 표본은 가능한 설명이지 이번 실험에서 각각 분리해 검증한 원인은 아니다. 국제전 기록까지 LCK 입력에 반영하는 효과 역시 이번에는 검증하지 않았다.

다음 후보: 여러 리그 사전학습 후 LCK fine-tuning, LCK 비중을 유지하는 sampling, 리그 강도 보정. 이들은 아직 실행하지 않았다. 이번 결과를 보고 2026년에 맞춰 계속 조정하면 새 독립 성능 검증이 되지 않는다. 배포 변경 없음.


In [5]:
display(pd.read_csv(RESULTS/'paired_comparison.csv'))
display(pd.read_csv(RESULTS/'monthly_2026.csv').round(4))
display(pd.read_csv(RESULTS/'seed_scores_2026.csv').round(4))
print(json.dumps(json.loads((RESULTS/'manifest.json').read_text()),indent=2))

,track,new_only_correct,old_only_correct,both_correct,both_wrong
0,cross_position,5,2,104,75
1,position_additive,1,7,111,67
2,upper_lower,3,7,108,68


,track,month,n,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,cross_position,2026-01,24,0.7083,0.7852,0.6598,0.2334,0.4400,0.5551
1,cross_position,2026-02,15,0.4000,0.2955,0.7031,0.2550,0.4374,0.5419
2,cross_position,2026-03,1,1.0000,NaN,0.6212,0.2141,0.4627,0.4627
3,cross_position,2026-04,44,0.6591,0.7158,0.6634,0.2352,0.4292,0.5767
4,cross_position,2026-05,46,0.7174,0.7942,0.6634,0.2351,0.4213,0.5733
5,cross_position,2026-06,5,0.4000,0.6667,0.6945,0.2508,0.4660,0.5856
6,cross_position,2026-07,6,0.5000,0.3333,0.6959,0.2514,0.4788,0.5206
7,cross_position,2026-08,38,0.4211,0.4696,0.6955,0.2512,0.4396,0.5647
8,cross_position,2026-09,7,0.2857,0.3000,0.6990,0.2530,0.4753,0.5601
9,position_additive,2026-01,24,0.6667,0.7926,0.6455,0.2263,0.4105,0.5907


,track,seed,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,position_additive,42,0.4624,0.4722,0.6952,0.2510,0.4570,0.5374
1,position_additive,43,0.5860,0.6277,0.6743,0.2407,0.3941,0.6352
2,position_additive,44,0.6183,0.6385,0.6650,0.2362,0.2623,0.7493
3,cross_position,42,0.6075,0.6592,0.6578,0.2328,0.3121,0.7252
4,cross_position,43,0.5968,0.6260,0.6854,0.2461,0.4582,0.5466
5,cross_position,44,0.5215,0.5720,0.6919,0.2494,0.4887,0.5082
6,upper_lower,42,0.6129,0.6592,0.6743,0.2406,0.4110,0.5813
7,upper_lower,43,0.5645,0.6396,0.6810,0.2440,0.4479,0.5651
8,upper_lower,44,0.6129,0.6230,0.6907,0.2488,0.4843,0.5166


{
  "source_sha256": {
    "2025": "c9a158b9e0a965a47d31d3674c127a26f75e6c91a324bd1858e4784b1336214a",
    "2026": "024330eb7a03e07c1aba55e17abf2e55f8e79e46e42e3f47389194173e59730a"
  },
  "checkpoint_sha256": "dd15fb5b3c79f26c2e01fe997d3fbe19918b60b19e2f7b0fe142af42b2bbe877",
  "checks": [
    "same LCK training inputs",
    "same 186 test series and labels",
    "no 2026 fitting",
    "team swap symmetry",
    "checkpoint unchanged"
  ],
  "raw_leagues": 45,
  "raw_games": 10041,
  "excluded_games": 1182
}
